In [2]:
# my_dashboard.ipynb
# Минималистичный интерфейс управления JetBot с работающей нейросетью

# ============================================
# 1. ИМПОРТЫ И ЗАГРУЗКА
# ============================================
%run my_camera.ipynb
%run my_basic.ipynb

import ipywidgets as widgets
from IPython.display import display
import cv2
import threading
import time
import numpy as np
from ipywidgets import Layout, VBox, HBox, Button, Image, FloatSlider, ToggleButton
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image as PILImage

# ============================================
# 2. НАСТРОЙКИ СТИЛЯ
# ============================================
style = """
<style>
    body {
        background-color: #1a1a1a;
        color: #e0e0e0;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif;
    }
    .dashboard {
        max-width: 1200px;
        margin: 0 auto;
        padding: 20px;
    }
    .video-container {
        background: #000000;
        border: 2px solid #333333;
        border-radius: 8px;
        padding: 4px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.5);
    }
    .control-panel {
        background: #000000;
        border: 2px solid #333333;
        border-radius: 8px;
        padding: 20px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.5);
    }
    .slider-label {
        color: #a0a0a0;
        font-size: 12px;
        letter-spacing: 0.5px;
        margin-top: 10px;
    }
    .button-minimal {
        background: #1a1a1a !important;
        color: #e0e0e0 !important;
        border: 2px solid #333333 !important;
        border-radius: 6px !important;
        font-size: 14px !important;
        font-weight: 400 !important;
        transition: all 0.2s ease;
    }
    .button-minimal:hover {
        background: #2a2a2a !important;
        border-color: #4a4a4a !important;
    }
    .button-minimal.danger {
        border-color: #663333 !important;
        color: #ff9999 !important;
    }
    .button-minimal.mode-off {
        border-color: #cc8800 !important;
        color: #ffaa66 !important;
    }
    .button-minimal.mode-on {
        border-color: #44aa44 !important;
        color: #88ff88 !important;
    }
    .stats-panel {
        background: #000000;
        border: 2px solid #333333;
        border-radius: 8px;
        padding: 15px;
        margin-top: 10px;
        font-size: 13px;
        color: #b0b0b0;
    }
    .frame-black {
        border: 2px solid #333333;
        border-radius: 8px;
        overflow: hidden;
        background: #000000;
    }
    h1, h2, h3 {
        color: #ffffff;
        font-weight: 300;
        letter-spacing: 0.5px;
    }
    hr {
        border: none;
        border-top: 1px solid #333333;
        margin: 15px 0;
    }
</style>
"""
display(widgets.HTML(style))

# ============================================
# 3. ЗАГРУЗКА НЕЙРОСЕТИ
# ============================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🤖 Используется устройство: {device}")

# Параметры из my_live_demo
MODEL_PATH = "/home/jetsonbot/jetbot/notebooks/my_teleop/my_model.pth"
FORWARD_SPEED = 0.3
TURN_SPEED = 0.2
PROB_THRESHOLD = 0.7
AUTO_MODE = False

# Загрузка модели
print("📦 Загрузка нейросети...")
try:
    model = models.resnet18(pretrained=False)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model = model.to(device)
    model.eval()
    print("✅ Нейросеть загружена")
    AUTO_AVAILABLE = True
except Exception as e:
    print(f"⚠️ Нейросеть не загружена: {e}")
    AUTO_AVAILABLE = False

# Трансформации для нейросети
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================
# 4. ЗАПУСК КАМЕРЫ
# ============================================
camera = Camera()
camera.start()
time.sleep(1)

# ============================================
# 5. ПЕРЕКЛЮЧАТЕЛЬ ЦВЕТОВ (на всякий случай)
# ============================================
color_mode = 'BGR2RGB'  # или 'NONE' или 'RGB2BGR'
color_toggle = widgets.ToggleButton(
    value=True,
    description='🎨 Цвет: BGR→RGB',
    layout=Layout(width='200px')
)
color_toggle.add_class('button-minimal')

def on_color_change(change):
    global color_mode
    if change['new']:
        color_mode = 'BGR2RGB'
        color_toggle.description = '🎨 Цвет: BGR→RGB'
    else:
        color_mode = 'NONE'
        color_toggle.description = '🎨 Цвет: BGR (без конвертации)'
color_toggle.observe(on_color_change, names='value')

# ============================================
# 6. СОЗДАНИЕ ВИДЖЕТОВ
# ============================================

# Видео с камеры
video_feed = widgets.Image(format='jpeg', width=640, height=480)
video_frame = widgets.VBox([video_feed], layout=Layout(padding='4px'))
video_frame.add_class('frame-black')

# Слайдеры
left_speed = widgets.FloatSlider(
    value=0.0, min=-0.8, max=0.8, step=0.05,
    layout=Layout(width='100%')
)

right_speed = widgets.FloatSlider(
    value=0.0, min=-0.8, max=0.8, step=0.05,
    layout=Layout(width='100%')
)

speed_control = widgets.FloatSlider(
    value=0.3, min=0.0, max=0.8, step=0.05,
    layout=Layout(width='100%')
)

# Кнопки управления
btn_forward = widgets.Button(description='↑', layout=Layout(width='60px', height='60px'))
btn_left = widgets.Button(description='←', layout=Layout(width='60px', height='60px'))
btn_stop = widgets.Button(description='⏹', layout=Layout(width='60px', height='60px'))
btn_right = widgets.Button(description='→', layout=Layout(width='60px', height='60px'))
btn_backward = widgets.Button(description='↓', layout=Layout(width='60px', height='60px'))

# Стили для кнопок
for btn in [btn_forward, btn_left, btn_right, btn_backward]:
    btn.add_class('button-minimal')
btn_stop.add_class('button-minimal')
btn_stop.add_class('danger')

# Переключатель режима
mode_toggle = widgets.ToggleButton(
    value=False,
    description='РУЧНОЙ',
    disabled=not AUTO_AVAILABLE,
    layout=Layout(width='120px')
)
mode_toggle.add_class('button-minimal')
mode_toggle.add_class('mode-off')

# Индикатор режима
mode_indicator = widgets.HTML(
    value="<span style='background:#333; color:#ffaa66; padding:4px 10px; border-radius:4px;'>⚡ РУЧНОЙ</span>"
)

# ============================================
# 7. ФУНКЦИИ УПРАВЛЕНИЯ
# ============================================
def update_video():
    """Обновление видеопотока с выбором цветокоррекции"""
    while True:
        frame = camera.read()
        if frame is not None:
            frame_resized = cv2.resize(frame, (640, 480))
            
            # Применяем выбранную цветокоррекцию
            if color_mode == 'BGR2RGB':
                frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
            elif color_mode == 'RGB2BGR':
                frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_RGB2BGR)
            else:
                frame_rgb = frame_resized
            
            _, jpeg = cv2.imencode('.jpg', frame_rgb, [cv2.IMWRITE_JPEG_QUALITY, 85])
            video_feed.value = jpeg.tobytes()
        time.sleep(0.03)

def auto_control():
    """Автономное управление с нейросетью (как в my_live_demo)"""
    global AUTO_MODE
    frame_count = 0
    
    while AUTO_MODE:
        frame = camera.read()
        if frame is not None:
            frame_count += 1
            
            if frame_count % 3 == 0:  # Каждый 3-й кадр
                # Подготовка изображения
                img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img_pil = PILImage.fromarray(img)
                img_tensor = transform(img_pil).unsqueeze(0).to(device)
                
                # Предсказание
                with torch.no_grad():
                    output = model(img_tensor)
                    probabilities = torch.nn.functional.softmax(output[0], dim=0)
                    blocked_prob = probabilities[1].item()
                
                # Управление
                if blocked_prob > PROB_THRESHOLD:
                    left_motor.value = -TURN_SPEED
                    right_motor.value = TURN_SPEED
                else:
                    left_motor.value = FORWARD_SPEED
                    right_motor.value = FORWARD_SPEED
                
                # Обновляем слайдеры для визуализации
                left_speed.value = left_motor.value
                right_speed.value = right_motor.value
        
        time.sleep(0.03)

def stop_robot(b=None):
    """Экстренная остановка"""
    global AUTO_MODE
    AUTO_MODE = False
    if AUTO_MODE:
        mode_toggle.value = False
    left_motor.value = 0
    right_motor.value = 0
    left_speed.value = 0
    right_speed.value = 0
    pcf.output_state = 0xFF
    pcf._write()
    print("⛔ Стоп")

def on_forward(b):
    if not AUTO_MODE:
        left_speed.value = speed_control.value
        right_speed.value = speed_control.value
        left_motor.value = left_speed.value
        right_motor.value = right_speed.value

def on_backward(b):
    if not AUTO_MODE:
        left_speed.value = -speed_control.value
        right_speed.value = -speed_control.value
        left_motor.value = left_speed.value
        right_motor.value = right_speed.value

def on_left(b):
    if not AUTO_MODE:
        left_speed.value = -speed_control.value
        right_speed.value = speed_control.value
        left_motor.value = left_speed.value
        right_motor.value = right_speed.value

def on_right(b):
    if not AUTO_MODE:
        left_speed.value = speed_control.value
        right_speed.value = -speed_control.value
        left_motor.value = left_speed.value
        right_motor.value = right_speed.value

def on_mode_change(change):
    global AUTO_MODE
    AUTO_MODE = change['new']
    if AUTO_MODE:
        mode_toggle.description = 'АВТО'
        mode_toggle.remove_class('mode-off')
        mode_toggle.add_class('mode-on')
        mode_indicator.value = "<span style='background:#333; color:#88ff88; padding:4px 10px; border-radius:4px;'>🤖 АВТО</span>"
        # Запуск автономного режима в отдельном потоке
        auto_thread = threading.Thread(target=auto_control)
        auto_thread.daemon = True
        auto_thread.start()
    else:
        mode_toggle.description = 'РУЧНОЙ'
        mode_toggle.remove_class('mode-on')
        mode_toggle.add_class('mode-off')
        mode_indicator.value = "<span style='background:#333; color:#ffaa66; padding:4px 10px; border-radius:4px;'>⚡ РУЧНОЙ</span>"
        stop_robot()

# Привязка событий
btn_forward.on_click(on_forward)
btn_backward.on_click(on_backward)
btn_left.on_click(on_left)
btn_right.on_click(on_right)
btn_stop.on_click(stop_robot)
mode_toggle.observe(on_mode_change, names='value')

def on_slider_change(change):
    if not AUTO_MODE:
        left_motor.value = left_speed.value
        right_motor.value = right_speed.value

left_speed.observe(on_slider_change, names='value')
right_speed.observe(on_slider_change, names='value')

# ============================================
# 8. СБОРКА ИНТЕРФЕЙСА
# ============================================

# Левая панель
left_panel = widgets.VBox([
    widgets.HTML("<h2>📹 КАМЕРА</h2>"),
    video_frame,
    color_toggle
], layout=Layout(width='660px'))

# Правая панель
control_panel = widgets.VBox([
    widgets.HTML("<h2>🎮 УПРАВЛЕНИЕ</h2>"),
    
    widgets.HBox([mode_toggle, mode_indicator], layout=Layout(justify_content='space-between')),
    widgets.HTML("<hr>"),
    
    widgets.HTML("<div class='slider-label'>ЛЕВОЕ КОЛЕСО</div>"),
    left_speed,
    widgets.HTML("<div class='slider-label'>ПРАВОЕ КОЛЕСО</div>"),
    right_speed,
    widgets.HTML("<div class='slider-label'>БАЗОВАЯ СКОРОСТЬ</div>"),
    speed_control,
    widgets.HTML("<hr>"),
    
    widgets.HTML("<div class='slider-label'>РУЧНОЕ УПРАВЛЕНИЕ</div>"),
    widgets.HBox([
        widgets.VBox([btn_forward], layout=Layout(align_items='center')),
        widgets.HBox([btn_left, btn_stop, btn_right]),
        widgets.VBox([btn_backward], layout=Layout(align_items='center'))
    ], layout=Layout(justify_content='center', gap='10px')),
    widgets.HTML("<hr>"),
    
    widgets.VBox([
        widgets.HTML(f"<b>Статус:</b> <span style='color:#4caf50;'>● ONLINE</span>"),
        widgets.HTML(f"<b>Нейросеть:</b> {'ЗАГРУЖЕНА' if AUTO_AVAILABLE else 'НЕДОСТУПНА'}"),
        widgets.HTML(f"<b>Порог:</b> {PROB_THRESHOLD}"),
    ], layout=Layout(padding='15px'))
    
], layout=Layout(width='500px', padding='20px'))
control_panel.add_class('control-panel')

# Главный контейнер
dashboard = widgets.VBox([
    widgets.HTML("<h1 style='text-align:center; font-weight:300;'>JETBOT CONTROL</h1>"),
    widgets.HBox([left_panel, control_panel], layout=Layout(justify_content='center', gap='20px'))
], layout=Layout(padding='20px'))

display(dashboard)

# ============================================
# 9. ЗАПУСК
# ============================================
video_thread = threading.Thread(target=update_video)
video_thread.daemon = True
video_thread.start()

print("\n✅ Dashboard запущен")
print("🎨 Переключай цвет кнопкой под видео")
print("🎮 Ручной режим: слайдеры/кнопки")
print("🤖 Автономный режим: нейросеть управляет")
print("⛔ Кнопка СТОП для остановки")

HTML(value="\n<style>\n    body {\n        background-color: #1a1a1a;\n        color: #e0e0e0;\n        font-f…

🤖 Используется устройство: cuda
📦 Загрузка нейросети...
✅ Нейросеть загружена
Попытка открыть камеру с конвейером:
nvarguscamerasrc ! video/x-raw(memory:NVMM),width=640,height=480,framerate=30/1 ! nvvidconv ! video/x-raw,format=BGRx ! videoconvert ! video/x-raw,format=BGR ! appsink drop=1
✅ Камера успешно открыта через GStreamer.
📷 Поток захвата кадров запущен



✅ Dashboard запущен
🎨 Переключай цвет кнопкой под видео
🎮 Ручной режим: слайдеры/кнопки
🤖 Автономный режим: нейросеть управляет
⛔ Кнопка СТОП для остановки
